<div style="text-align: center; background: #000; padding: 20px 10px; border-radius: 15px; box-shadow: 0 10px 25px rgba(0,0,0,0.7); font-family: 'Segoe UI', sans-serif; color: white;">

  <img src="https://i.postimg.cc/VN0mKkBT/images.jpg" 
       alt="Title Image" 
       style="
       display: block;
       width: 100%;
       height: auto;
       margin: 2px auto 10px auto;
       border-radius: 10px;
       box-shadow: 0 0 10px #666;
       ">
  
  <span style="
      color: #B8860B;
      font-family: 'Times New Roman', Times, serif;
      font-weight: bold;
      padding-bottom: 2px;
  ">
      Samane Najarian
  </span>

</div>

In [1]:
# import Libraries
import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
#!pip install ydata_profiling
from ydata_profiling import ProfileReport
from sklearn.model_selection import train_test_split
from scipy.stats import skew
import matplotlib.pyplot as plt
import seaborn as sns
import matplotlib.patheffects as pe
from sklearn.preprocessing import PowerTransformer
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, average_precision_score
from sklearn.metrics import confusion_matrix, ConfusionMatrixDisplay
from sklearn.tree import DecisionTreeClassifier

/tmp/ipykernel_58/2764326442.py:5: DeprecationWarning: 
    `import ydata_profiling` is deprecated and will not receive more updates. 
    Please install fg-data-profiling via `pip install fg-data-profiling` and use `import data_profiling` instead.
    
  from ydata_profiling import ProfileReport


In [2]:
# Load the credit card dataset and display the first five rows
Credit_Card = pd.read_excel(
    "/kaggle/input/datasets/samanenajarian/default-of-credit-card-clients/default of credit card clients.xls",
    header=1
)

Credit_Card.head()

,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_0,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,default payment next month
0,1,20000,2,2,1,24,2,2,-1,-1,...,0,0,0,0,689,0,0,0,0,1
1,2,120000,2,2,2,26,-1,2,0,0,...,3272,3455,3261,0,1000,1000,1000,0,2000,1
2,3,90000,2,2,2,34,0,0,0,0,...,14331,14948,15549,1518,1500,1000,1000,1000,5000,0
3,4,50000,2,2,1,37,0,0,0,0,...,28314,28959,29547,2000,2019,1200,1100,1069,1000,0
4,5,50000,1,2,1,57,-1,0,-1,0,...,20940,19146,19131,2000,36681,10000,9000,689,679,0


In [3]:
# Display dataset structure, column names, data types, and non-null counts
Credit_Card.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 30000 entries, 0 to 29999
Data columns (total 25 columns):
 #   Column                      Non-Null Count  Dtype
---  ------                      --------------  -----
 0   ID                          30000 non-null  int64
 1   LIMIT_BAL                   30000 non-null  int64
 2   SEX                         30000 non-null  int64
 3   EDUCATION                   30000 non-null  int64
 4   MARRIAGE                    30000 non-null  int64
 5   AGE                         30000 non-null  int64
 6   PAY_0                       30000 non-null  int64
 7   PAY_2                       30000 non-null  int64
 8   PAY_3                       30000 non-null  int64
 9   PAY_4                       30000 non-null  int64
 10  PAY_5                       30000 non-null  int64
 11  PAY_6                       30000 non-null  int64
 12  BILL_AMT1                   30000 non-null  int64
 13  BILL_AMT2                   30000 non-null  int64
 14  BILL_A

In [4]:
Credit_Card.isna().sum()

ID                            0
LIMIT_BAL                     0
SEX                           0
EDUCATION                     0
MARRIAGE                      0
AGE                           0
PAY_0                         0
PAY_2                         0
PAY_3                         0
PAY_4                         0
PAY_5                         0
PAY_6                         0
BILL_AMT1                     0
BILL_AMT2                     0
BILL_AMT3                     0
BILL_AMT4                     0
BILL_AMT5                     0
BILL_AMT6                     0
PAY_AMT1                      0
PAY_AMT2                      0
PAY_AMT3                      0
PAY_AMT4                      0
PAY_AMT5                      0
PAY_AMT6                      0
default payment next month    0
dtype: int64

In [ ]:
# Display the number of rows and columns in the dataset
print(f"Shape of data:", Credit_Card.shape)

In [ ]:
pay_cols = ['PAY_0', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6']

for col in pay_cols:
    Credit_Card[col] = Credit_Card[col].astype('category')

In [ ]:
# Check for duplicated rows in the dataset
Credit_Card.duplicated().sum()

In [ ]:
# Set the customer ID as the DataFrame index
Credit_Card.set_index("ID", inplace=True)

In [ ]:
# Generate and save an automated exploratory data analysis report
#Profile_Credit_Card = ProfileReport(
    #Credit_Card,
    #title="Default of Credit Card Clients"
#)

#Profile_Credit_Card.to_file("Default of Credit Card Clients.html")

In [ ]:
# Replace numerical demographic codes with meaningful category labels
Credit_Card["EDUCATION"] = Credit_Card["EDUCATION"].replace({
    1: "Graduate school",
    2: "University",
    3: "High school",
    4: "Other",
    0: "Other",
    5: "Other",
    6: "Other"
})

Credit_Card["MARRIAGE"] = Credit_Card["MARRIAGE"].replace({
    1: "Married",
    2: "Single",
    3: "Other",
    0: "Other"
})

Credit_Card["SEX"] = Credit_Card["SEX"].replace({
    1: "Male",
    2: "Female"
})

Credit_Card.head()

<div style="
    text-align: center;
    background: linear-gradient(90deg, #111827, #1f2937);
    padding: 15px;
    border-radius: 10px;
    box-shadow: 0 5px 15px rgba(0,0,0,0.3);
    font-family: 'Segoe UI', sans-serif;
">

  <h2 style="
      color: #FBBF24;
      font-family: 'Georgia', serif;
      font-weight: bold;
      margin: 0;
      letter-spacing: 1px;
  ">
      Descriptive Statisrics
  </h2>

</div>

<div style="
    background-color:#FFFFFF;
    border-left:6px solid #1F3A5F;
    padding:22px;
    border-radius:10px;
    margin:20px 0;
    box-shadow:5px 5px 14px rgba(100,100,100,0.25);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:24px;
">
Q1 — What does the portfolio look like before modeling?
</h2>

<p style="
    color:#333333;
    font-size:16px;
    line-height:1.7;
">
Describe the number of clients, default/non-default class distribution,
credit-limit distribution, age profile, repayment-status variables and
recent bill/payment behavior. Identify unusual encodings or values that
require documentation.
</p>

</div>

In [ ]:
# Check the number of clients and the distribution of default and non-default customers
print("Number of clients:", len(Credit_Card))

print("\nDefault distribution:")
print(Credit_Card["default payment next month"].value_counts())

print("\nDefault percentage:")
print(
    Credit_Card["default payment next month"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

<div style="
    background-color:#FFFFFF;
    padding:18px 22px;
    margin:20px 0;
    border-radius:8px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="color:#1F3A5F; margin-top:0;">
Portfolio Size & Default Rate
</h3>

<p style="font-size:15px; line-height:1.8; color:#333333;">
The portfolio consists of <b>30,000 clients</b>.
<b>23,364 clients (77.88%)</b> are classified as non-default,
whereas <b>6,636 clients (22.12%)</b> are classified as default.
</p>

<table style="
    width:70%;
    border-collapse:collapse;
    margin:15px 0;
    font-size:14px;
">
<tr style="background-color:#1F3A5F; color:white;">
    <th style="padding:10px; text-align:left;">Class</th>
    <th style="padding:10px; text-align:right;">Customers</th>
    <th style="padding:10px; text-align:right;">Percentage</th>
</tr>
<tr>
    <td style="padding:9px; border-bottom:1px solid #ddd;">Non-default</td>
    <td style="padding:9px; border-bottom:1px solid #ddd; text-align:right;">23,364</td>
    <td style="padding:9px; border-bottom:1px solid #ddd; text-align:right;">77.88%</td>
</tr>
<tr>
    <td style="padding:9px;">Default</td>
    <td style="padding:9px; text-align:right;">6,636</td>
    <td style="padding:9px; text-align:right;">22.12%</td>
</tr>
</table>

<p style="font-size:15px; line-height:1.8; color:#333333;">
The target is therefore <b>moderately imbalanced</b>, with non-default
customers representing the majority class. This imbalance should be
considered during model development because accuracy alone may not
adequately reflect the model's ability to identify defaulting customers.
</p>

</div>

In [ ]:
# Summarize the distribution of customers' credit limits
Credit_Card["LIMIT_BAL"].describe()

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    Credit_Card["LIMIT_BAL"],
    bins=30,
    color="#009B77",
    edgecolor="#555555",
    linewidth=1
)

plt.xlabel("Credit Limit", fontsize=12, fontweight="bold")
plt.ylabel("Number of Customers", fontsize=12, fontweight="bold")
plt.title("Distribution of Credit Limits", fontsize=15, fontweight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:18px 22px;
    margin:20px 0;
    border-radius:8px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="color:#1F3A5F; margin-top:0;">
Credit-Limit Distribution
</h3>

<p style="font-size:15px; line-height:1.8; color:#333333;">
The credit limit (<b>LIMIT_BAL</b>) varies substantially across the
30,000 clients. The average credit limit is approximately
<b>167,484</b>, while the median is <b>140,000</b>. The standard
deviation of approximately <b>129,748</b> indicates considerable
variation in customers' assigned credit limits.
</p>

<p style="font-size:15px; line-height:1.8; color:#333333;">
The middle 50% of customers have credit limits between
<b>50,000</b> (25th percentile) and <b>240,000</b> (75th percentile).
The observed values range from <b>10,000</b> to <b>1,000,000</b>,
showing that the portfolio includes customers with both relatively
small and very large credit limits.
</p>

<p style="font-size:15px; line-height:1.8; color:#333333;">
The mean being higher than the median suggests a <b>right-skewed
distribution</b>, which is consistent with the presence of customers
with substantially higher credit limits. This variable therefore
shows considerable heterogeneity across the portfolio and should be
examined further during preprocessing.
</p>

</div>

In [ ]:
# Summarize the age distribution of customers
Credit_Card["AGE"].describe()

In [ ]:
plt.figure(figsize=(9, 5))

plt.hist(
    Credit_Card["AGE"],
    bins=25,
    color="#009B77",
    edgecolor="#555555",
    linewidth=1
)

plt.xlabel("Age", fontsize=12, fontweight="bold")
plt.ylabel("Number of Customers", fontsize=12, fontweight="bold")
plt.title("Age Distribution of Customers", fontsize=15, fontweight="bold")

plt.grid(axis="y", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The age profile shows that the clients are relatively concentrated
around their mid-thirties. The <b>mean age is 35.49 years</b>, while the
<b>median is 34 years</b>. The middle 50% of clients are between
<b>28 and 41 years old</b>, indicating that most customers fall within
a relatively narrow age range. Overall, ages range from
<b>21 to 79 years</b>, with a standard deviation of approximately
<b>9.22 years</b>, showing moderate variation in the age of clients.
    The histogram confirms that customers are predominantly concentrated
in the <b>20–50 age range</b>, with the highest concentration around
the early-to-mid thirties. The distribution gradually decreases at
older ages, with relatively few customers above 60. No obvious
implausible ages are visible in the distribution, and the observed
range of <b>21 to 79 years</b> appears reasonable for a credit-card
client portfolio.
</p>

</div>

In [ ]:
# Examine the repayment-status variables over the previous six months
pay_cols = [
    "PAY_0", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

Credit_Card[pay_cols].describe()

In [ ]:
# Check the encoded values used in the repayment-status variables
for col in pay_cols:
    print(Credit_Card[col].value_counts().sort_index())
    print("-"*30)

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:20px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Repayment Status
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The repayment-status variables (<b>PAY_0–PAY_6</b>) describe customers'
repayment behavior over the previous six months. The variables contain
values ranging from <b>-2 to 8</b>, with <b>0</b> being the most common
category across all six months. For example, PAY_0 contains 14,737
observations with a value of 0, compared with 5,686 observations with
-1 and 2,759 with -2.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Higher repayment-status codes are much less frequent. In PAY_0,
values of 3 or above account for only a small proportion of customers,
and a similar pattern is observed for the earlier months. This
suggests that severe repayment delays are relatively uncommon in the
portfolio.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;


</div>

In [ ]:
# Summarize recent payment amounts
pay_amt_cols = [
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

print("Pay amounts:")
display(Credit_Card[pay_amt_cols].describe())


<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Recent Payment Behavior
</h3>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Payment amounts are generally much smaller than bill amounts. The
median payment ranges from approximately <b>1,500 to 2,100</b> across
the six months, while the means are higher, ranging from approximately
<b>4,799 to 5,921</b>. This difference indicates that payment amounts
are strongly influenced by customers making substantially larger
payments.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Payment amounts also include <b>zero values</b>, indicating that some
customers made no recorded payment during a given month. The maximum
payment amounts are considerably higher than the corresponding medians,
further indicating substantial variation in payment behavior.
</p>

</div>

In [ ]:
# Summarize recent bill amounts
bill_cols = [
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6"
]

print("Bill amounts:")
display(Credit_Card[bill_cols].describe())


<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Recent Bill Behavior
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The bill amounts show a declining pattern across the six observed
months. The average bill amount decreases from approximately
<b>51,223</b> in BILL_AMT1 to <b>38,872</b> in BILL_AMT6, while the
median decreases from approximately <b>22,382</b> to <b>17,071</b>.
This suggests that customers' outstanding bill amounts were generally
lower in the later months.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Bill amounts also show substantial variability. For example, BILL_AMT1
ranges from <b>-165,580</b> to <b>964,511</b>, while BILL_AMT6 ranges
from <b>-339,603</b> to <b>961,664</b>. The presence of negative bill
amounts and the large difference between the median and maximum values
should be documented and examined during data-quality and preprocessing
steps.
</p>

In [ ]:
# Examine the encoded categorical values for potential unusual or undocumented categories
for col in ["SEX", "EDUCATION", "MARRIAGE"]:
    print(Credit_Card[col].value_counts().sort_index())
    print("-"*30)

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Categorical Variables
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The categorical variables show a clear distribution across the
customer portfolio. The dataset contains <b>18,112 female</b> and
<b>11,888 male</b> clients. For education, <b>14,030</b> clients have
a university education, <b>10,585</b> have graduate-school education,
and <b>4,917</b> have a high-school education. A relatively small group
of <b>468 clients</b> falls into the <b>Other</b> category.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Regarding marital status, <b>15,964</b> clients are single and
<b>13,659</b> are married, while <b>377</b> observations are classified
as <b>Other</b>.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The original dataset contains numerical codes for these categorical
variables, including several non-standard education and marriage
codes. These codes were consolidated into meaningful categories before
modeling. In particular, education codes <b>0, 4, 5, and 6</b> were
grouped into <b>Other</b>, while marriage code <b>0</b> was also treated
as <b>Other</b>. This makes the categorical variables easier to
interpret and avoids treating nominal categories as numerical
quantities.
</p>

</div>

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
">
Q2 — Which customer characteristics differ between default and non-default groups?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Compare selected variables such as credit limit, repayment status, bill
amounts and payment amounts across the target groups. Use plots and
summary tables that directly support interpretation.
</p>

</div>

In [ ]:
limit_comparison = (
    Credit_Card.groupby("default payment next month")["LIMIT_BAL"]
    .agg(["mean", "median", "std"])
    .T
)

limit_comparison.columns = ["Non-default", "Default"]

limit_comparison

In [ ]:
sns.boxplot(
    data=Credit_Card,
    x="default payment next month",
    y="LIMIT_BAL"
)

plt.xlabel("Default Status")
plt.ylabel("Credit Limit")
plt.title("Credit Limit by Default Status")
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Credit Limit by Default Status
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows that customers who did not default generally have
higher credit limits than those who defaulted. The mean credit limit is
<b>178,100</b> for non-default customers, compared with
<b>130,110</b> for default customers. The difference is also evident in
the median, with <b>150,000</b> for non-default customers and
<b>90,000</b> for default customers.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Both groups show substantial variation in credit limits, with standard
deviations of <b>131,628</b> for non-default customers and
<b>115,379</b> for default customers. Overall, the results suggest that
defaulting customers tend to have <b>lower credit limits</b> than
non-default customers. However, the relatively large variation within
both groups indicates considerable overlap between them.
</p>

</div>


In [ ]:
for col in pay_cols:

    table = pd.crosstab(
        Credit_Card[col],
        Credit_Card["default payment next month"],
        normalize="columns"
    ) * 100

    table.columns = ["Non-default", "Default"]

    print(f"\n{col}")
    display(table.round(2))

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 10))

for col, ax in zip(pay_cols, axes.flatten()):

    table = pd.crosstab(
        Credit_Card[col],
        Credit_Card["default payment next month"],
        normalize="columns"
    ) * 100

    table.columns = ["Non-default", "Default"]

    bars = table.plot(
        kind="bar",
        ax=ax,
        color=["#B8D8D8", "#F4B6C2"],
        edgecolor="#555555",
        linewidth=1.1
    )

    # Bar shadows
    for patch in ax.patches:
        patch.set_path_effects([
            pe.withSimplePatchShadow(
                offset=(2, -2),
                alpha=0.18
            ),
            pe.Normal()
        ])

    ax.set_title(
        f"{col} Distribution by Default Status",
        fontsize=13,
        fontweight="bold",
        pad=10
    )

    ax.set_xlabel("Repayment Status", fontsize=11, fontweight="bold")
    ax.set_ylabel("Percentage (%)", fontsize=11, fontweight="bold")

    ax.tick_params(axis="x", rotation=0, labelsize=9)
    ax.tick_params(axis="y", labelsize=9)

    ax.grid(axis="y", alpha=0.2)
    ax.set_axisbelow(True)

    ax.legend(
        title="Target",
        frameon=True,
        edgecolor="#555555"
    )

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Repayment Status by Default Status
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows clear differences in repayment-status distributions
between non-default and default customers across all repayment periods.
Among non-default customers, repayment status <b>0</b> is the most common
category, accounting for approximately <b>55%–59%</b> of customers across
the different periods. For default customers, the proportion in status
<b>0</b> is considerably lower, ranging from <b>28.45%</b> in PAY_0 to
<b>48.15%</b> in PAY_5.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
In contrast, default customers show substantially higher proportions in
repayment status <b>2</b>. For example, status 2 accounts for
<b>32.91%</b> of default customers in PAY_2, compared with only
<b>7.46%</b> of non-default customers. A similar pattern is observed
across PAY_0, PAY_3, PAY_4, PAY_5, and PAY_6. Higher repayment-status
categories are also generally more common among default customers,
although their overall proportions are relatively small.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, the results suggest that <b>default customers tend to have more
delayed repayment statuses</b>, while non-default customers are more
concentrated in status 0. This indicates that repayment status may provide
useful information for distinguishing between default and non-default
customers.
</p>

</div>


In [ ]:
bill_comparison = (
    Credit_Card.groupby("default payment next month")[bill_cols]
    .mean()
    .T
)

bill_comparison.columns = ["Non-default", "Default"]

bill_comparison

In [ ]:
bill_means = (
    Credit_Card.groupby("default payment next month")[bill_cols]
    .mean()
    .T
)

bill_means.columns = ["Non-default", "Default"]

ax = bill_means.plot(
    kind="bar",
    figsize=(11, 6),
    color=["#B8D8D8", "#F4B6C2"],   # pastel colors
    edgecolor="#555555",             # border around bars
    linewidth=1.2
)

# Add subtle shadow effect
for patch in ax.patches:
    shadow = plt.Rectangle(
        (patch.get_x() + 0.03, patch.get_y() - 0.01),
        patch.get_width(),
        patch.get_height(),
        color="gray",
        alpha=0.15,
        zorder=0
    )
    ax.add_patch(shadow)

plt.xlabel("Bill Amount", fontsize=12, fontweight="bold")
plt.ylabel("Mean Amount", fontsize=12, fontweight="bold")
plt.title(
    "Mean Bill Amounts by Default Status",
    fontsize=15,
    fontweight="bold",
    pad=15
)

plt.xticks(rotation=0, fontsize=11)
plt.yticks(fontsize=10)

plt.legend(
    title="Target",
    frameon=True,
    edgecolor="#555555"
)

plt.grid(axis="y", alpha=0.2)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Bill Amounts by Default Status
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows that non-default customers generally have slightly
higher mean bill amounts than default customers across all six billing
periods. The mean bill amount for non-default customers decreases from
<b>51,994</b> in BILL_AMT1 to <b>39,042</b> in BILL_AMT6. Similarly,
the mean bill amount for default customers decreases from
<b>48,509</b> to <b>38,271</b> over the same periods.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The difference between the two groups is relatively small compared with
the overall bill amounts. The largest difference occurs in
<b>BILL_AMT1</b>, where non-default customers have a mean bill amount of
<b>51,994</b> compared with <b>48,509</b> for default customers. The
difference becomes smaller in the later periods, reaching only about
<b>771</b> in BILL_AMT6.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, the results suggest that <b>bill amounts alone show only modest
differences between default and non-default customers</b>. Although
non-default customers consistently have somewhat higher mean bill
amounts, the relatively small differences suggest that bill amounts may
be less discriminative than repayment status.
</p>

</div>


In [ ]:
pay_comparison = (
    Credit_Card.groupby("default payment next month")[pay_amt_cols]
    .mean()
    .T
)

pay_comparison.columns = ["Non-default", "Default"]

pay_comparison

In [ ]:
payment_means = (
    Credit_Card.groupby("default payment next month")[pay_amt_cols]
    .mean()
    .T
)

payment_means.columns = ["Non-default", "Default"]

ax = payment_means.plot(
    kind="bar",
    figsize=(11, 6),
    color=["#B8D8D8", "#F4B6C2"],   # pastel colors
    edgecolor="#555555",             # border around bars
    linewidth=1.2
)

# Add subtle shadow effect
for patch in ax.patches:
    shadow = plt.Rectangle(
        (patch.get_x() + 0.03, patch.get_y() - 0.01),
        patch.get_width(),
        patch.get_height(),
        color="gray",
        alpha=0.15,
        zorder=0
    )
    ax.add_patch(shadow)

plt.xlabel("Payment Amount", fontsize=12, fontweight="bold")
plt.ylabel("Mean Amount", fontsize=12, fontweight="bold")
plt.title(
    "Mean Payment Amounts by Default Status",
    fontsize=15,
    fontweight="bold",
    pad=15
)

plt.xticks(rotation=0, fontsize=11)
plt.yticks(fontsize=10)

plt.legend(
    title="Target",
    frameon=True,
    edgecolor="#555555"
)

plt.grid(axis="y", alpha=0.2)
plt.gca().set_axisbelow(True)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Payment Amounts by Default Status
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows that non-default customers have substantially higher
mean payment amounts than default customers across all six payment
periods. The mean payment amount for non-default customers ranges from
<b>5,248</b> to <b>6,640</b>, whereas for default customers it ranges
from <b>3,156</b> to <b>3,441</b>.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The largest difference is observed in <b>PAY_AMT2</b>, where non-default
customers have a mean payment amount of <b>6,640</b>, compared with
<b>3,389</b> for default customers. A similar pattern is observed across
the remaining payment periods, with non-default customers consistently
making considerably larger payments.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, the results suggest that <b>payment amounts provide a clearer
distinction between default and non-default customers</b> than bill
amounts. Non-default customers consistently make higher payments, while
default customers show substantially lower mean payment amounts across
all observed periods.
</p>

</div>


<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
">
Q3 — Which recent repayment behaviors appear most informative?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Focus on repayment-status history and recent payment/bill patterns.
Explain which variables show the clearest separation between the two
outcome groups without claiming causation.
</p>

</div>

In [ ]:
# Identify customers with delayed repayment (status >= 2)

delayed_repayment = {}

for col in pay_cols:
    
    delayed_rate = (
        Credit_Card.groupby("default payment next month")[col]
        .apply(lambda x: (x.astype(int) >= 2).mean() * 100)
    )
    
    delayed_repayment[col] = (
        delayed_rate.loc[1] - delayed_rate.loc[0]
    )

delayed_repayment = (
    pd.Series(delayed_repayment)
    .sort_values(ascending=False)
)

print("Difference in Delayed Repayment Rate:")
display(
    delayed_repayment
    .to_frame("Difference (%)")
    .round(2)
)

In [ ]:
plt.figure(figsize=(9, 5))

delayed_repayment.sort_values().plot(
    kind="barh",
    color="#C5B4E3",
    edgecolor="#555555",
    linewidth=1
)

plt.xlabel(
    "Difference in Delayed Repayment Rate (percentage points)",
    fontsize=12
)
plt.ylabel("Repayment Status Variable", fontsize=12)

plt.title(
    "Separation in Delayed Repayment Across Default Groups",
    fontsize=14,
    fontweight="bold"
)

plt.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Payment Amount Separation
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows a clear difference in mean payment amounts between
default and non-default customers across all six payment periods. The
largest relative difference occurs in <b>PAY_AMT2</b>, at
<b>48.97%</b>, followed by <b>PAY_AMT1</b> at <b>46.14%</b>.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The remaining payment variables also show substantial differences,
with relative differences ranging from <b>38.66%</b> in PAY_AMT5 to
<b>41.47%</b> in PAY_AMT3. This indicates that the separation is
consistent across the different payment periods rather than being
limited to a single period.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, the results suggest that <b>payment amounts provide relatively
clear separation between default and non-default customers</b>.
In particular, the more recent payment periods, especially
<b>PAY_AMT2 and PAY_AMT1</b>, show the largest differences between the
two outcome groups. These results indicate an association with the
outcome, without implying causation.
</p>

</div>

In [ ]:
payment_means = (
    Credit_Card.groupby("default payment next month")[pay_amt_cols]
    .mean()
)

payment_separation = (
    (payment_means.loc[0] - payment_means.loc[1])
    / payment_means.loc[0] * 100
)

print("Relative Difference in Payment Amounts:")
display(
    payment_separation
    .sort_values(ascending=False)
    .to_frame("Relative Difference (%)")
    .round(2)
)

In [ ]:
plt.figure(figsize=(9, 5))

payment_separation.sort_values().plot(
    kind="barh",
    color="#C5B4E3",
    edgecolor="#555555",
    linewidth=1
)

plt.xlabel(
    "Relative Difference in Mean Payment Amount (%)",
    fontsize=12
)
plt.ylabel("Payment Amount Variable", fontsize=12)

plt.title(
    "Separation in Payment Amounts Across Default Groups",
    fontsize=14,
    fontweight="bold"
)

plt.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Payment Amount Separation
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows a clear and consistent difference in payment
amounts between default and non-default customers across all six
payment periods. The largest relative difference occurs in
<b>PAY_AMT2</b> (<b>48.97%</b>), followed by <b>PAY_AMT1</b>
(<b>46.14%</b>).
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The remaining variables also show substantial separation, with relative
differences of <b>41.47%</b> for PAY_AMT3, <b>40.47%</b> for PAY_AMT4,
<b>39.83%</b> for PAY_AMT6, and <b>38.66%</b> for PAY_AMT5. Thus, the
difference between the two groups is consistently large across the
payment history rather than being concentrated in a single period.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, <b>payment amounts provide clear separation between default and
non-default customers</b>. In particular, PAY_AMT2 and PAY_AMT1 show
the largest relative differences, while the other payment variables
also provide substantial separation. These results indicate an
association with the outcome and do not imply that payment amounts
cause default.
</p>

</div>


In [ ]:
bill_means = (
    Credit_Card.groupby("default payment next month")[bill_cols]
    .mean()
)

bill_separation = (
    (bill_means.loc[0] - bill_means.loc[1])
    / bill_means.loc[0] * 100
)

print("Relative Difference in Bill Amounts:")
display(
    bill_separation
    .sort_values(ascending=False)
    .to_frame("Relative Difference (%)")
    .round(2)
)

In [ ]:
plt.figure(figsize=(9, 5))

bill_separation.sort_values().plot(
    kind="barh",
    color="#C5B4E3",
    edgecolor="#555555",
    linewidth=1
)

plt.xlabel(
    "Relative Difference in Mean Bill Amount (%)",
    fontsize=12
)
plt.ylabel("Bill Amount Variable", fontsize=12)

plt.title(
    "Separation in Bill Amounts Across Default Groups",
    fontsize=14,
    fontweight="bold"
)

plt.grid(axis="x", linestyle="--", alpha=0.3)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:4px 4px 12px rgba(100,100,100,0.18);
    font-family:Georgia, serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Bill Amount Separation
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The comparison shows relatively small differences in bill amounts between
default and non-default customers across all six billing periods. The
largest relative difference occurs in <b>BILL_AMT1</b> (<b>6.70%</b>),
followed by <b>BILL_AMT3</b> (<b>4.95%</b>) and <b>BILL_AMT2</b>
(<b>4.90%</b>).
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The remaining variables show even smaller differences, with relative
differences of <b>3.61%</b> for BILL_AMT4, <b>2.44%</b> for BILL_AMT5,
and <b>1.97%</b> for BILL_AMT6. This indicates that bill amounts provide
limited separation between the two outcome groups compared with the
payment amount variables.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, <b>bill amounts show relatively weak separation between default
and non-default customers</b>. BILL_AMT1 provides the clearest difference,
while the separation becomes smaller for the later billing periods.
These results indicate an association with the outcome and do not imply
that bill amounts cause default.
</p>

</div>


<div style="
    text-align: center;
    background: linear-gradient(90deg, #111827, #1f2937);
    padding: 15px;
    border-radius: 10px;
    box-shadow: 0 5px 15px rgba(0,0,0,0.3);
    font-family: 'Segoe UI', sans-serif;
">

  <h2 style="
      color: #FBBF24;
      font-family: 'Georgia', serif;
      font-weight: bold;
      margin: 0;
      letter-spacing: 1px;
  ">
      Exploratory Data Analysis (EDA)
  </h2>

</div>

In [ ]:
# Define the continuous numerical variables for analysis and modeling
continuous_cols = [
    "LIMIT_BAL",
    "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

In [ ]:
# Separate the predictor variables (X) from the target variable (y)
X = Credit_Card.drop("default payment next month", axis=1)
y = Credit_Card["default payment next month"]

In [ ]:
# Split the data into stratified training, validation, and test sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

X_train, X_val, y_train, y_val = train_test_split(
    X_train, y_train,
    test_size=0.25,
    random_state=42,
    stratify=y_train
)

print(f"X_train:", X_train.shape, f"y_train:", y_train.shape, "\n" 
      f"X_test:", X_test.shape, f"y_test:", y_test.shape, "\n"
      f"X_validation:", X_val.shape, f"y_Validation:", y_val.shape)

In [ ]:
outlier_summary = []

for col in continuous_cols:
    Q1 = X_train[col].quantile(0.25)
    Q3 = X_train[col].quantile(0.75)
    
    IQR = Q3 - Q1
    
    lower_bound = Q1 - 1.5 * IQR
    upper_bound = Q3 + 1.5 * IQR
    
    outliers = X_train[(X_train[col] < lower_bound) | (X_train[col] > upper_bound)]
    
    outlier_summary.append({
        'Feature': col,
        'Number of Outliers': len(outliers),
        'Outlier Percentage (%)': round(len(outliers)/len(Credit_Card)*100,2)
    })

outlier_table = pd.DataFrame(outlier_summary)

outlier_table.sort_values(
    'Outlier Percentage (%)',
    ascending=False
)

### Outlier Analysis

Outliers were detected using the Interquartile Range (IQR) method on continuous 
financial variables. The highest proportion of outliers was observed in payment 
amount features (PAY_AMT1–PAY_AMT6) and bill amount features (BILL_AMT1–BILL_AMT6). 
These extreme values are expected in credit card data because some customers may 
have significantly higher payment activity or outstanding balances.

Since these observations may represent genuine customer behavior rather than 
data errors, they were retained for further modeling. Appropriate modeling 
techniques will be used to reduce the potential influence of extreme values.

A small number of potential outliers were also detected in **AGE** and **LIMIT_BAL**. These observations were examined and considered plausible values rather than clear data-entry errors. Older customers and customers with relatively high credit limits are possible within the population and do not necessarily indicate abnormal observations. Therefore, these values were also retained for further modeling.

In [ ]:
# Select continuous numerical features
continuous_features = X_train.select_dtypes(include=['int64', 'float64']).columns

# Calculate skewness
skewness = X_train[continuous_features].apply(skew)

# Sort from highest to lowest skewness
skewness = skewness.sort_values(ascending=False)

In [ ]:
skewness_table = pd.DataFrame({
    'Feature': skewness.index,
    'Skewness': skewness.values
})

print(skewness_table)

In [ ]:
# Set general style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.facecolor'] = '#F7F7F7'
plt.rcParams['figure.facecolor'] = 'white'

financial_cols = [
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
    'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
    'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))

for ax, col in zip(axes.flat, financial_cols):

    ax.hist(
        X_train[col],
        bins=30,
        color='#6A5ACD',
        edgecolor='#333333',
        linewidth=0.7
    )

    ax.set_title(
        col,
        fontsize=12,
        fontweight='bold'
    )

    ax.set_xlabel(
        col,
        fontsize=10
    )

    ax.set_ylabel(
        'Frequency',
        fontsize=10
    )

    ax.tick_params(
        axis='both',
        labelsize=9
    )

    ax.grid(
        axis='y',
        linestyle='--',
        alpha=0.3
    )

# Hide unused subplot
for ax in axes.flat[len(financial_cols):]:
    ax.set_visible(False)

plt.suptitle(
    'Distribution of Financial Variables',
    fontsize=18,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AGE
axes[0].hist(
    X_train['AGE'],
    bins=30,
    color='#6A5ACD',
    edgecolor='#333333',
    linewidth=0.7
)

axes[0].set_title('Age Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# LIMIT_BAL
axes[1].hist(
    X_train['LIMIT_BAL'],
    bins=30,
    color='#6A5ACD',
    edgecolor='#333333',
    linewidth=0.7
)

axes[1].set_title('Credit Limit Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('LIMIT_BAL', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

plt.suptitle(
    'Distribution of Age and Credit Limit',
    fontsize=17,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

In [ ]:
# ============================================================
# Create a copy of the ORIGINAL X_train
# ============================================================

X_train_transformed = X_train.copy()


# ============================================================
# LIMIT_BAL and AGE → Yeo-Johnson
# ============================================================

yeo_cols = [
    'LIMIT_BAL',
    'AGE'
]

yeo_johnson = PowerTransformer(
    method='yeo-johnson',
    standardize=False
)

X_train_transformed[yeo_cols] = yeo_johnson.fit_transform(
    X_train[yeo_cols]
)


# ============================================================
# BILL_AMT → Signed Square Root
# ============================================================

bill_cols = [
    'BILL_AMT1',
    'BILL_AMT2',
    'BILL_AMT3',
    'BILL_AMT4',
    'BILL_AMT5',
    'BILL_AMT6'
]

for col in bill_cols:
    X_train_transformed[col] = (
        np.sign(X_train[col]) *
        np.sqrt(np.abs(X_train[col]))
    )


# ============================================================
# PAY_AMT → Log transformation
# ============================================================

pay_cols = [
    'PAY_AMT1',
    'PAY_AMT2',
    'PAY_AMT3',
    'PAY_AMT4',
    'PAY_AMT5',
    'PAY_AMT6'
]

for col in pay_cols:
    X_train_transformed[col] = np.log1p(
        X_train[col]
    )


# ============================================================
# Calculate skewness AFTER transformation
# ============================================================

skewness_after = (
    X_train_transformed[continuous_cols]
    .skew()
    .sort_values(ascending=False)
)

print("Skewness AFTER transformation:")
skewness_after

In [ ]:
X_train_transformed.head()

In [ ]:
# Set general style
plt.rcParams['font.family'] = 'DejaVu Sans'
plt.rcParams['axes.facecolor'] = '#F7F7F7'
plt.rcParams['figure.facecolor'] = 'white'

financial_cols = [
    'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3',
    'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6',
    'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3',
    'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6'
]

fig, axes = plt.subplots(4, 4, figsize=(16, 14))

for ax, col in zip(axes.flat, financial_cols):

    ax.hist(
        X_train_transformed[col],
        bins=30,
        color='#6A5ACD',
        edgecolor='#333333',
        linewidth=0.7
    )

    ax.set_title(
        col,
        fontsize=12,
        fontweight='bold'
    )

    ax.set_xlabel(
        col,
        fontsize=10
    )

    ax.set_ylabel(
        'Frequency',
        fontsize=10
    )

    ax.tick_params(
        axis='both',
        labelsize=9
    )

    ax.grid(
        axis='y',
        linestyle='--',
        alpha=0.3
    )

# Hide unused subplot
for ax in axes.flat[len(financial_cols):]:
    ax.set_visible(False)

plt.suptitle(
    'Distribution of Financial Variables',
    fontsize=18,
    fontweight='bold',
    y=1.02
)

plt.tight_layout()
plt.show()

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# AGE
axes[0].hist(
    X_train_transformed['AGE'],
    bins=30,
    color='#6A5ACD',
    edgecolor='#333333',
    linewidth=0.7
)

axes[0].set_title('Age Distribution', fontsize=13, fontweight='bold')
axes[0].set_xlabel('Age', fontsize=11)
axes[0].set_ylabel('Frequency', fontsize=11)
axes[0].grid(axis='y', linestyle='--', alpha=0.3)

# LIMIT_BAL
axes[1].hist(
    X_train_transformed['LIMIT_BAL'],
    bins=30,
    color='#6A5ACD',
    edgecolor='#333333',
    linewidth=0.7
)

axes[1].set_title('Credit Limit Distribution', fontsize=13, fontweight='bold')
axes[1].set_xlabel('LIMIT_BAL', fontsize=11)
axes[1].set_ylabel('Frequency', fontsize=11)
axes[1].grid(axis='y', linestyle='--', alpha=0.3)

plt.suptitle(
    'Distribution of Age and Credit Limit',
    fontsize=17,
    fontweight='bold'
)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
">
Q4 — Can a simple baseline model predict default risk?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Build Logistic Regression as the required baseline. Use a reproducible
train/test workflow and fit preprocessing only on the training data.
A shallow Decision Tree may be added as an optional comparison.
</p>

</div>

In [ ]:
# Define categorical, ordinal, and numerical variables for preprocessing
categorical_cols = ["SEX", "EDUCATION", "MARRIAGE"]

ordinal_cols = [
    "PAY_0", "PAY_2", "PAY_3",
    "PAY_4", "PAY_5", "PAY_6"
]

numerical_cols = [
    "LIMIT_BAL", "AGE",
    "BILL_AMT1", "BILL_AMT2", "BILL_AMT3",
    "BILL_AMT4", "BILL_AMT5", "BILL_AMT6",
    "PAY_AMT1", "PAY_AMT2", "PAY_AMT3",
    "PAY_AMT4", "PAY_AMT5", "PAY_AMT6"
]

In [ ]:
# Create a preprocessing pipeline for categorical, ordinal, and numerical variables
preprocessor = ColumnTransformer(
    transformers=[
        ("categorical", OneHotEncoder(handle_unknown="ignore"), categorical_cols),
        ("ordinal", StandardScaler(), ordinal_cols),
        ("numerical", StandardScaler(), numerical_cols)
    ]
)

In [ ]:
# Build and fit a Logistic Regression pipeline with preprocessing and class weighting
logistic_pipeline = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", LogisticRegression(
            max_iter=1000,
            class_weight="balanced"
        ))
    ]
)

logistic_pipeline.fit(X_train, y_train)

In [ ]:
# Generate binary default predictions for the test set
y_pred_Logistic = logistic_pipeline.predict(X_test)

y_pred_Logistic

In [ ]:
# Generate predicted probabilities of default for the test set
y_prob_Logistic = logistic_pipeline.predict_proba(X_test)[:, 1]

y_prob_Logistic

In [ ]:
# Evaluation

Logistic_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Score": [
        accuracy_score(y_test, y_pred_Logistic),
        precision_score(y_test, y_pred_Logistic, zero_division=0),
        recall_score(y_test, y_pred_Logistic, zero_division=0),
        f1_score(y_test, y_pred_Logistic, zero_division=0),
        roc_auc_score(y_test, y_prob_Logistic),
        average_precision_score(y_test, y_prob_Logistic)
    ]
})

Logistic_results["Score"] = Logistic_results["Score"].round(4)

display(Logistic_results)

In [ ]:
cm_Logistic = confusion_matrix(y_test, y_pred_Logistic)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_Logistic,
    display_labels=["No Default", "Default"]
)

disp.plot()

plt.title("Confusion Matrix - Logistic Regression")
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Baseline Logistic Regression Performance
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The Logistic Regression baseline achieves an <b>accuracy of 67.57%</b>
on the test set. Its <b>ROC-AUC of 0.712</b> indicates that the model
has a moderate ability to distinguish between customers who default
and those who do not.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The model achieves a <b>recall of 63.53%</b>, meaning that it identifies
a relatively large proportion of the actual default cases. However,
the <b>precision is 36.57%</b>, indicating that a considerable number
of customers predicted as defaulters do not actually default. The
resulting <b>F1-score of 0.464</b> reflects this trade-off between
precision and recall.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The <b>PR-AUC of 0.493</b> provides additional information about the
model's performance on the positive default class, particularly given
the imbalance between default and non-default customers. Overall, the
baseline demonstrates that the available customer characteristics
contain useful predictive information, while also leaving substantial
room for improvement.
</p>

</div>

In [ ]:
# Shallow Decision Tree

tree_model = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("classifier", DecisionTreeClassifier(
            max_depth=3,
            random_state=42,
            class_weight="balanced"
        ))
    ]
)


In [ ]:
# Fit ONLY on training data

tree_model.fit(X_train, y_train)

In [ ]:
# 3. Predictions

y_pred_tree = tree_model.predict(X_test)
y_prob_tree = tree_model.predict_proba(X_test)[:, 1]

In [ ]:
# Evaluation

tree_results = pd.DataFrame({
    "Metric": [
        "Accuracy",
        "Precision",
        "Recall",
        "F1-score",
        "ROC-AUC",
        "PR-AUC"
    ],
    "Score": [
        accuracy_score(y_test, y_pred_tree),
        precision_score(y_test, y_pred_tree, zero_division=0),
        recall_score(y_test, y_pred_tree, zero_division=0),
        f1_score(y_test, y_pred_tree, zero_division=0),
        roc_auc_score(y_test, y_prob_tree),
        average_precision_score(y_test, y_prob_tree)
    ]
})

tree_results["Score"] = tree_results["Score"].round(4)

display(tree_results)

In [ ]:
cm_Tree = confusion_matrix(y_test, y_pred_tree)

disp = ConfusionMatrixDisplay(
    confusion_matrix=cm_Tree,
    display_labels=["No Default", "Default"]
)

disp.plot()

plt.title("Confusion Matrix - Logistic Regression")
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
">
Q5 — Why is Accuracy not enough?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Evaluate the classifier with a confusion matrix, Precision, Recall,
F1-score, ROC-AUC and PR-AUC. Explain what each metric reveals and why
imbalanced classification requires more than one metric.
</p>

</div>

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Accuracy alone is not sufficient for evaluating the classifiers because the dataset is imbalanced, with non-default customers representing the majority of observations. Therefore, a model may achieve reasonable accuracy while still failing to identify a substantial number of actual default cases.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The <b>confusion matrix</b> provides a detailed view of the predictions by showing true positives, true negatives, false positives, and false negatives. <b>Precision</b> indicates how reliable the positive (default) predictions are, while <b>Recall</b> measures the model's ability to identify customers who actually default. The <b>F1-score</b> combines precision and recall and provides a balanced measure when both are important.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
<b>ROC-AUC</b> evaluates the ability of the classifiers to distinguish between default and non-default customers across different classification thresholds. <b>PR-AUC</b> focuses more directly on the positive class and is particularly useful when the classes are imbalanced.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, these measures provide complementary information about model performance. While accuracy describes overall correctness, the other metrics reveal how well the classifiers identify the minority class and the types of errors they make. Therefore, an imbalanced classification problem requires <b>multiple evaluation metrics</b> rather than relying on accuracy alone.
</p>

</div>


<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:20px;
">
Q6 — Which error matters more for the business?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Discuss false negatives (a defaulting client predicted as non-default)
and false positives (a non-defaulting client flagged as risky). Explain
the business trade-off and which metric/threshold you would monitor.
</p>

</div>

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
In this credit-default setting, <b>false negatives</b> are likely to
represent the more costly error. A false negative occurs when a
customer who eventually defaults is classified as non-default. This
may expose the business to additional credit losses because the
customer's risk is not identified in advance.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
<b>False positives</b> also have a business cost. These occur when a
non-defaulting customer is incorrectly flagged as risky and may result
in unnecessary credit restrictions, additional monitoring, or lost
business opportunities. Therefore, reducing false negatives should not
come at the expense of generating an excessive number of false
positives.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Because missing a potential default is considered more costly, the
primary metric to monitor should be <b>Recall</b>, while
<b>Precision</b> should be monitored alongside it to control the number
of false-positive classifications. The classification threshold should
therefore be selected based on the desired balance between these two
types of error rather than simply using the default threshold of 0.50.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Overall, the preferred operating point should prioritize a sufficiently
high <b>Recall</b> while maintaining an acceptable level of
<b>Precision</b>. The final threshold should ultimately reflect the
business cost assigned to false negatives versus false positives.
</p>

</div>

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:20px;
">
Q7 — Can model probabilities become useful risk groups?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Use predicted default probabilities to create Low, Medium and High risk
groups. Choose thresholds using validation results and business
reasoning rather than arbitrary labels. Report default prevalence and
customer count in each group.
</p>

</div>

In [ ]:
# Generate predicted default probabilities for the validation set
y_prob_val = logistic_pipeline.predict_proba(X_val)[:, 1]

y_prob_val

In [ ]:
# Summarize the distribution of predicted default probabilities on the validation set
pd.Series(y_prob_val).describe()

In [ ]:
plt.rcParams["font.family"] = "DejaVu Serif"

fig, ax = plt.subplots(figsize=(9, 5.5))

# Histogram of predicted default probabilities
ax.hist(
    y_prob_val,
    bins=20,
    color="#009B77",
    edgecolor="#555555",
    linewidth=1.2,
    alpha=0.85
)

# Add a subtle gray shadow
ax.patch.set_path_effects([
    pe.SimplePatchShadow(
        offset=(4, -4),
        shadow_rgbFace="gray",
        alpha=0.25
    ),
    pe.Normal()
])

# Labels
ax.set_xlabel(
    "Predicted Probability of Default",
    fontsize=13,
    fontweight="bold"
)

ax.set_ylabel(
    "Number of Customers",
    fontsize=13,
    fontweight="bold"
)

# Title
ax.set_title(
    "Distribution of Predicted Default Probabilities",
    fontsize=16,
    fontweight="bold",
    pad=15
)

# Grid
ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.3
)

ax.tick_params(axis="both", labelsize=11)

plt.tight_layout()
plt.show()

In [ ]:
# Divide validation customers into probability bands and calculate the observed default rate in each band
bins = [0, 0.2, 0.3, 0.4, 0.5, 0.6, 0.7, 1]

val_risk = pd.DataFrame({
    "probability": y_prob_val,
    "actual_default": y_val.values
})

val_risk["probability_band"] = pd.cut(
    val_risk["probability"],
    bins=bins,
    include_lowest=True
)

val_risk.groupby("probability_band", observed=True)["actual_default"].agg(
    ["count", "mean"]
)

In [ ]:
# Define risk groups based on predicted default probability
def assign_risk(probability):
    if probability < 0.5:
        return "Low Risk"
    elif probability < 0.7:
        return "Medium Risk"
    else:
        return "High Risk"

In [ ]:
# Create a risk-group table using predicted probabilities and actual default outcomes
risk_groups = pd.DataFrame({
    "Predicted_Probability": y_prob_val,
    "Actual_Default": y_val.values
})

# Assign each customer to Low, Medium, or High Risk
risk_groups["Risk_Group"] = risk_groups["Predicted_Probability"].apply(
    assign_risk
)

In [ ]:
# Summarize the number of customers and actual default rate in each risk group
risk_summary = (
    risk_groups.groupby("Risk_Group")
    .agg(
        Customer_Count=("Actual_Default", "size"),
        Default_Rate=("Actual_Default", "mean")
    )
    .reset_index()
    .sort_values(by="Default_Rate", ascending=True)
)

risk_summary

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:20px;
">
Q8 — What variables appear to drive model predictions?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Interpret standardized Logistic Regression coefficients or another
simple explainability output. Separate model association from causation
and discuss correlated predictors.
</p>

</div>

In [ ]:
feature_names = logistic_pipeline.named_steps[
    "preprocessor"
].get_feature_names_out()

In [ ]:
coefficients = logistic_pipeline.named_steps[
    "classifier"
].coef_[0]

In [ ]:
coefficient_table = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": coefficients,
    "Absolute_Coefficient": np.abs(coefficients)
})

coefficient_table = coefficient_table.sort_values(
    by="Absolute_Coefficient",
    ascending=False
)

coefficient_table.head(15)

In [ ]:
top_coefficients = (
    coefficient_table
    .head(15)
    .sort_values("Coefficient")
)

plt.figure(figsize=(10, 7))

plt.barh(
    top_coefficients["Feature"],
    top_coefficients["Coefficient"],
    edgecolor="#555555",
    linewidth=1
)

plt.axvline(
    0,
    color="#555555",
    linewidth=1
)

plt.xlabel(
    "Standardized Logistic Regression Coefficient",
    fontsize=12,
    fontweight="bold"
)

plt.ylabel(
    "Feature",
    fontsize=12,
    fontweight="bold"
)

plt.title(
    "Top Features Influencing Logistic Regression Predictions",
    fontsize=14,
    fontweight="bold"
)

plt.grid(
    axis="x",
    linestyle="--",
    alpha=0.3
)

plt.tight_layout()
plt.show()

In [ ]:
positive_coefficients = (
    coefficient_table[
        coefficient_table["Coefficient"] > 0
    ]
    .sort_values("Coefficient", ascending=False)
    .head(10)
)

negative_coefficients = (
    coefficient_table[
        coefficient_table["Coefficient"] < 0
    ]
    .sort_values("Coefficient")
    .head(10)
)

positive_coefficients

In [ ]:
negative_coefficients

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:20px;
">
Q9 — How stable is the chosen threshold?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Compare at least two plausible classification thresholds and show how
Precision/Recall or confusion-matrix counts change. Explain why
threshold choice is a business decision as well as a statistical
decision.
</p>

</div>

In [ ]:
# Evaluate Precision, Recall, and F1-score across different classification thresholds
thresholds = np.arange(0.1, 0.95, 0.05)

results_val = []

for threshold in thresholds:
    
    y_pred_val = (y_prob_val >= threshold).astype(int)
    
    results_val.append({
        "Threshold": threshold,
        "Precision": precision_score(y_val, y_pred_val),
        "Recall": recall_score(y_val, y_pred_val),
        "F1-score": f1_score(y_val, y_pred_val)
    })

threshold_results_val = pd.DataFrame(results_val)

threshold_results_val

In [ ]:
# Identify the classification threshold that achieves the highest F1-score on the validation set
best_threshold = threshold_results_val.loc[
    threshold_results_val["F1-score"].idxmax(),
    "Threshold"
]

best_f1 = threshold_results_val["F1-score"].max()

print("Best threshold:", best_threshold)
print("Best F1-score:", best_f1)

In [ ]:
# Visualize the trade-off between Precision, Recall, and F1-score across classification thresholds
fig, ax = plt.subplots(figsize=(10, 6))

shadow = plt.Rectangle(
    (0.03, -0.03),
    0.94,
    1.0,
    transform=ax.transAxes,
    facecolor="gray",
    alpha=0.30,
    zorder=0
)

ax.add_patch(shadow)

# White plotting area
ax.set_facecolor("white")

# Plot the metrics
ax.plot(
    threshold_results_val["Threshold"],
    threshold_results_val["Precision"],
    color="#2563EB",
    marker="o",
    linewidth=2.5,
    markersize=6,
    label="Precision"
)

ax.plot(
    threshold_results_val["Threshold"],
    threshold_results_val["Recall"],
    color="#DC2626",
    marker="o",
    linewidth=2.5,
    markersize=6,
    label="Recall"
)

ax.plot(
    threshold_results_val["Threshold"],
    threshold_results_val["F1-score"],
    color="#059669",
    marker="o",
    linewidth=2.5,
    markersize=6,
    label="F1-score"
)

# Highlight the best threshold
ax.axvline(
    best_threshold,
    color="#7C3AED",
    linestyle="--",
    linewidth=2,
    label=f"Best Threshold = {best_threshold:.2f}"
)

# Labels
ax.set_xlabel(
    "Classification Threshold",
    fontsize=12,
    fontweight="bold"
)

ax.set_ylabel(
    "Score",
    fontsize=12,
    fontweight="bold"
)

# Title
ax.set_title(
    "Model Performance Across Classification Thresholds",
    fontsize=16,
    fontweight="bold",
    pad=15
)

# Grid
ax.grid(
    axis="y",
    linestyle="--",
    linewidth=0.7,
    alpha=0.3
)

# Legend
ax.legend(
    fontsize=10,
    frameon=True,
    shadow=True,
    fancybox=True
)

ax.tick_params(axis="both", labelsize=10)

plt.tight_layout()
plt.show()

<div style="
    background-color:#FFFFFF;
    padding:20px 24px;
    margin:25px 0;
    border-left:6px solid #1F3A5F;
    border-radius:8px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h2 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:20px;
">
Q10 — What should management do with the result?
</h2>

<p style="
    color:#333333;
    font-size:15px;
    line-height:1.8;
">
Provide a concise management summary: which customers/segments require
closer review, what evidence supports that conclusion, what the model
cannot establish, and what additional data or validation would be
required before real deployment.
</p>

</div>

<div style="
    background-color:#FFFFFF;
    padding:16px 22px;
    margin:15px 0 25px 0;
    border-left:4px solid #1F3A5F;
    border-radius:6px;
    box-shadow:5px 5px 14px rgba(100,100,100,0.20);
    font-family:Georgia, 'Times New Roman', serif;
">

<h3 style="
    color:#1F3A5F;
    margin-top:0;
    font-size:18px;
">
Management Summary
</h3>

<p style="
    margin:0;
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Management should give closer attention to customers classified as <b>High Risk</b>. This group contains 730 customers and has an observed default rate of <b>64.5%</b>, compared with 26.0% for Medium Risk and 12.2% for Low Risk customers. The increasing default rates across the probability bands also support the model's ability to distinguish customers with different levels of default risk.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The results provide useful evidence for identifying customers who may require further credit review or monitoring. The threshold analysis also shows that a threshold of <b>0.55</b> gives the highest F1-score (<b>0.514</b>), providing a reasonable balance between Precision and Recall. However, the models still have limitations in predictive performance and make both false-positive and false-negative predictions.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
The models <b>cannot establish that a particular customer will default or that any variable causes default</b>. They only estimate default risk based on patterns in the available historical data. Therefore, model predictions should support, rather than replace, professional credit assessment.
</p>

<p style="
    font-size:15px;
    line-height:1.8;
    color:#333333;
">
Before deployment, the model should be validated on new and representative data and evaluated for stability, fairness, calibration, and performance over time. Additional information such as <b>updated payment behaviour, credit history, debt and income information, and other relevant financial indicators</b> could also improve the reliability of the risk assessment.
</p>

</div>
